# OLS Return Prediction Model

This notebook implements an **Ordinary Least Squares (OLS)** model to predict **next-month stock excess returns (`y_next`)** using the **Fama-French five factors**.

The workflow mirrors the Random Forest model in the project:

1. Load the cleaned dataset.
2. Restrict predictions to **post-2022 data**.
3. Use a **36-month rolling training window**.
4. Estimate an OLS model each month.
5. Predict next-month returns.
6. Rank stocks by predicted return within each month.
7. Select the **Top 6 stocks per month** for portfolio construction.

In [8]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# =========================
# Load cleaned dataset
# =========================
df_model = pd.read_csv("../data_process/Data_clean.csv")

# Convert date column
df_model["MthCalDt"] = pd.to_datetime(df_model["MthCalDt"])

print("Original df_model shape:", df_model.shape)
print(
    "Original date range:", df_model["MthCalDt"].min(), "to", df_model["MthCalDt"].max()
)

display(df_model.head())

Original df_model shape: (1048575, 22)
Original date range: 1996-09-30 00:00:00 to 2025-12-31 00:00:00


/var/folders/rq/6gkwkw8n79d9pw5r3y76nfqc0000gn/T/ipykernel_41617/175599620.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_model["MthCalDt"] = pd.to_datetime(df_model["MthCalDt"])


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthRet,MthVol,ShrOut,DateKey,MthPrc_Abs,Date,Mkt-RF,SMB,HML,RMW,CMA,RF,DollarVol,ExcessRet,y_next,history_len
0,10001,36720410,EWST,7953,1997-12-31,9.0000,0.041502,111877,2395,199712,9.0000,199712,0.0136,-0.0209,0.0374,0.0064,0.0195,0.0048,1006893.000,0.036702,-0.004300,36
1,10001,36720410,EWST,7953,1998-01-30,9.0000,0.000000,68436,2395,199801,9.0000,199801,0.0014,-0.0128,-0.0168,0.0091,-0.0062,0.0043,615924.000,-0.004300,-0.010844,37
2,10001,36720410,EWST,7953,1998-02-27,8.9375,-0.006944,14128,2395,199802,8.9375,199802,0.0704,0.0005,-0.0093,-0.0037,-0.0258,0.0039,126269.000,-0.010844,-0.012702,38
3,10001,36720410,EWST,7953,1998-03-31,8.7500,-0.008802,77324,2401,199803,8.7500,199803,0.0476,-0.0073,0.0138,-0.0055,-0.0023,0.0039,676585.000,-0.012702,0.002843,39
4,10001,36720410,EWST,7953,1998-04-30,8.8125,0.007143,38426,2401,199804,8.8125,199804,0.0074,0.0001,0.0103,-0.0168,-0.0036,0.0043,338629.125,0.002843,-0.018184,40


In [9]:
# =========================
# Keep only post-2022 data
# =========================
df_model_post2022 = df_model[df_model["MthCalDt"] >= "2022-01-01"].copy()

print("Post-2022 df shape:", df_model_post2022.shape)
print(
    "Post-2022 date range:",
    df_model_post2022["MthCalDt"].min(),
    "to",
    df_model_post2022["MthCalDt"].max(),
)

# Feature columns
feature_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]

# Target column
target_col = "y_next"

# All available months
all_months_post2022 = sorted(df_model_post2022["DateKey"].unique())

print("Number of months:", len(all_months_post2022))
print("First months:", all_months_post2022[:5])
print("Last months:", all_months_post2022[-5:])

Post-2022 df shape: (189287, 22)
Post-2022 date range: 2022-01-31 00:00:00 to 2025-12-31 00:00:00
Number of months: 48
First months: [np.int64(202201), np.int64(202202), np.int64(202203), np.int64(202204), np.int64(202205)]
Last months: [np.int64(202508), np.int64(202509), np.int64(202510), np.int64(202511), np.int64(202512)]


In [13]:
# =========================
# Rolling 36-month OLS
# =========================

results = []

months = sorted(df_model["DateKey"].unique())

for i in range(36, len(months)):

    month = months[i]

    # previous 36 months
    train_months = months[i - 36 : i]

    train_data = df_model[df_model["DateKey"].isin(train_months)]
    test_data = df_model[df_model["DateKey"] == month]

    if len(train_data) == 0 or len(test_data) == 0:
        continue

    X_train = train_data[feature_cols]
    y_train = train_data[target_col]

    X_test = test_data[feature_cols]

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    temp = test_data.copy()
    temp["predicted_return"] = preds

    results.append(temp)

pred_df = pd.concat(results)

print("Prediction dataset shape:", pred_df.shape)
display(pred_df.head())

Prediction dataset shape: (956240, 23)


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthRet,MthVol,ShrOut,DateKey,MthPrc_Abs,Date,Mkt-RF,SMB,HML,RMW,CMA,RF,DollarVol,ExcessRet,y_next,history_len,predicted_return
21,10001,36720410,EWST,7953,1999-09-30,8.00000,-0.073439,74555,2450,199909,8.00000,199909,-0.0277,0.0248,-0.029,-0.0096,-0.0099,0.0039,5.964400e+05,-0.077339,0.082038,57,-0.002631
258,10002,05978R10,SABC,7954,1999-09-30,14.90625,0.154573,123582,7729,199909,14.90625,199909,-0.0277,0.0248,-0.029,-0.0096,-0.0099,0.0039,1.842144e+06,0.150673,-0.115011,57,-0.002631
395,10009,46334710,IROQ,7965,1999-09-30,17.25000,-0.061224,102841,2427,199909,17.25000,199909,-0.0277,0.0248,-0.029,-0.0096,-0.0099,0.0039,1.774007e+06,-0.065124,0.025086,57,-0.002631
432,10016,81002230,SCTT,1728,1999-09-30,19.75000,0.041186,1913107,17869,199909,19.75000,199909,-0.0277,0.0248,-0.029,-0.0096,-0.0099,0.0039,3.778386e+07,0.037286,-0.041875,57,-0.002631
494,10025,00103110,AEPI,7975,1999-09-30,37.00000,-0.038961,116011,7392,199909,37.00000,199909,-0.0277,0.0248,-0.029,-0.0096,-0.0099,0.0039,4.292407e+06,-0.042861,-0.139035,58,-0.002631


In [11]:
# =========================
# Rank stocks each month
# =========================

pred_df["rank"] = pred_df.groupby("DateKey")["predicted_return"].rank(ascending=False)

# Select Top 6 stocks per month
top6_ols = pred_df[pred_df["rank"] <= 6].copy()

print("Top-6 dataset shape:", top6_ols.shape)

display(top6_ols.head(20))

Top-6 dataset shape: (0, 24)


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthRet,MthVol,ShrOut,DateKey,MthPrc_Abs,Date,Mkt-RF,SMB,HML,RMW,CMA,RF,DollarVol,ExcessRet,y_next,history_len,predicted_return,rank


In [12]:
# =========================
# Evaluate model accuracy
# =========================

mse = mean_squared_error(pred_df["y_next"], pred_df["predicted_return"])
mae = mean_absolute_error(pred_df["y_next"], pred_df["predicted_return"])

print("OLS Performance")
print("MSE:", mse)
print("MAE:", mae)

OLS Performance
MSE: 0.01643502726004329
MAE: 0.07179562739121441


In [14]:
# =========================
# Save OLS selections
# =========================

top6_ols.to_csv("../data/ols_top6_by_month_post2022.csv", index=False)

print("OLS Top-6 selections saved.")

OLS Top-6 selections saved.
